# 6주차. 카나메이트가 약속을 결정하다

**부제:** Supervisor/Sub-agent 최종 일정 결정: Nana/Kana delegate로 팀 회의 가능 시간 확정

## 0. 목표

이번 주에는 supervisor가 업무 tool을 직접 호출하지 않고 `nana_agent`, `kana_agent` delegate tool만 호출하는 최종 구조를 LangChain agent로 검증한다.

성취 기준:

- 프록시 토큰을 사용하는 supervisor/sub-agent 흐름을 실행할 수 있다.
- Nana와 Kana의 내부 tool trace를 delegate payload에서 확인할 수 있다.
- “팀원 A/B/C와 다음 주 회의 시간을 잡아줘” 요청의 최종 후보 결정 근거를 설명할 수 있다.

![sub_agent](imgs/sub_agent.jpg)

![multi_agent](imgs/multi_agent.jpg)

## 1. 준비

6주차도 프록시 서버를 통해 모델 API를 호출한다. Supervisor, Nana sub-agent, Kana sub-agent가 모두 LangChain `create_agent` 기반으로 실행된다.

In [1]:
from __future__ import annotations

import json
import os
import sqlite3
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import dotenv_values, load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_openai import ChatOpenAI


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
# 이 노트북에서 쓰는 프록시/모델 설정값은 repo 루트의 .env 파일에서만 읽는다.
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)
ENV_VALUES = dotenv_values(ENV_PATH)


def required_env(name: str) -> str:
    value = (ENV_VALUES.get(name) or "").strip()
    if not value or value == "여기에 api key 입력":
        raise RuntimeError(f"repo 루트 .env 파일에 {name}을 설정한 뒤 다시 실행하세요.")
    return value


PROXY_TOKEN = required_env("PROXY_TOKEN")
PROXY_URL = required_env("PROXY_URL")
EMBEDDING_PROXY_URL = required_env("EMBEDDING_PROXY_URL")
OPENAI_MODEL = required_env("OPENAI_MODEL")
OPENAI_EMBEDDING_MODEL = required_env("OPENAI_EMBEDDING_MODEL")


def make_model(max_tokens: int = 700) -> ChatOpenAI:
    return ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=PROXY_TOKEN,
        base_url=PROXY_URL,
        temperature=0,
        max_completion_tokens=max_tokens,
    )


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace: list[dict[str, Any]] = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


def extract_delegate_payloads(supervisor_trace: list[dict[str, Any]]) -> list[dict[str, Any]]:
    payloads: list[dict[str, Any]] = []
    for event in supervisor_trace:
        if event["event"] == "tool_result" and event["tool_name"] in {"nana_agent", "kana_agent"}:
            payloads.append(json.loads(event["content"]))
    return payloads


class DisableParallelToolCalls(AgentMiddleware):
    def wrap_model_call(self, request, handler):
        request.model_settings["parallel_tool_calls"] = False
        return handler(request)

my_schedules = [
    {"title": "개인 운동", "date": "2026-05-20", "start_time": "10:00", "end_time": "11:00"}
]
external_member_slots = [
    {"member": "A", "date": "다음 주 화요일", "start_time": "15:00"},
    {"member": "B", "date": "다음 주 화요일", "start_time": "15:00"},
    {"member": "C", "date": "다음 주 화요일", "start_time": "15:00"},
]
show_json({"model": OPENAI_MODEL, "golden_request": "팀원 A/B/C와 다음 주 회의 시간을 잡아줘"})

{
  "model": "openai/gpt-4.1-mini",
  "golden_request": "팀원 A/B/C와 다음 주 회의 시간을 잡아줘"
}


## 2. 개념

Supervisor는 직접 개인 일정 tool이나 MCP 검색 tool을 호출하지 않는다. Nana는 내 개인 일정과 개인 RAG를 담당하고, Kana는 MCP SQLite에서 다른 사람들의 일정과 이전 대화 기록을 불러와 가능한 시간을 종합한다.

## 3. 기본 개념 실습

Nana/Kana 내부에서 사용할 업무 tool을 정의한다.

In [2]:
@tool("personal_list_schedules", description="내 개인 일정 목록을 조회한다.")
def personal_list_schedules(date_range: str = "next_week") -> str:
    """List my personal schedules."""
    return json.dumps({"date_range": date_range, "schedules": my_schedules}, ensure_ascii=False)


@tool("search_previous_conversations", description="MCP SQLite에서 팀원들의 이전 일정 대화를 검색한다.")
def search_previous_conversations(members: list[str]) -> str:
    """Search external member history through MCP SQLite."""
    return json.dumps({"members": members, "conversation_ids": [f"conv-{member.lower()}" for member in members]}, ensure_ascii=False)


@tool("extract_schedules_from_history", description="이전 대화에서 팀원별 가능 시간을 추출한다.")
def extract_schedules_from_history(conversation_ids: list[str]) -> str:
    """Extract member schedule candidates from history."""
    return json.dumps({"schedules": external_member_slots, "conversation_ids": conversation_ids}, ensure_ascii=False)


@tool("decide_final_slot", description="팀원 가능 시간 후보만 비교해 최종 회의 시간 후보를 결정한다. 개인 일정 충돌은 판단하지 않는다.")
def decide_final_slot(candidate_slots: list[str]) -> str:
    """Decide the final meeting slot."""
    return json.dumps({"final_slot": "다음 주 화요일 15:00", "external_reason": "A/B/C 가능 시간이 모두 겹치는 후보", "candidates": candidate_slots}, ensure_ascii=False)

show_json({"nana_tools": ["personal_list_schedules"], "kana_tools": ["search_previous_conversations", "extract_schedules_from_history", "decide_final_slot"]})

{
  "nana_tools": [
    "personal_list_schedules"
  ],
  "kana_tools": [
    "search_previous_conversations",
    "extract_schedules_from_history",
    "decide_final_slot"
  ]
}


## 4. 카나메이트 확장 예제

Delegate tool은 sub-agent를 실행하고, sub-agent 내부 trace를 JSON payload로 supervisor에게 돌려준다.

In [3]:
@tool("nana_agent", description="내 개인 일정 생성/조회/삭제와 개인 RAG 검색을 Nana sub-agent에게 위임한다.")
def nana_agent(request: str) -> str:
    """Delegate personal schedule work to Nana."""
    agent = create_agent(
        model=make_model(700),
        tools=[personal_list_schedules],
        middleware=[DisableParallelToolCalls()],
        system_prompt=("너는 개인 메이트 Nana다. 어떤 회의 요청이든 반드시 personal_list_schedules tool을 먼저 호출해 " "내 개인 일정 충돌 여부를 확인한 뒤 답한다. tool 호출 없이 답하지 않는다."),
    )
    result = agent.invoke({"messages": [{"role": "user", "content": request}]})
    return json.dumps({"agent": "nana", "answer": final_text(result), "trace": extract_tool_trace(result)}, ensure_ascii=False)


@tool("kana_agent", description="MCP SQLite에서 다른 사람들의 일정과 이전 대화 기록을 불러와 팀원 기준 회의 시간 후보를 결정한다.")
def kana_agent(request: str) -> str:
    """Delegate group scheduling work to Kana."""
    agent = create_agent(
        model=make_model(900),
        tools=[search_previous_conversations, extract_schedules_from_history, decide_final_slot],
        middleware=[DisableParallelToolCalls()],
        system_prompt=(
            "너는 그룹 메이트 Kana다. 팀원 A/B/C 회의 요청을 받으면 반드시 "
            "search_previous_conversations, extract_schedules_from_history, decide_final_slot을 이 순서로 모두 호출해 팀원 가능 시간 후보만 결정한다. "
            "개인 일정 정보가 요청에 포함되어도 사용하지 않는다. "
            "답변에는 개인 일정, 개인 일정 충돌 여부, 충돌하지 않는다는 단정을 쓰지 말고 팀원 후보와 근거만 작성한다. "
            "tool 호출 없이 답하지 않는다."
        ),
    )
    result = agent.invoke({"messages": [{"role": "user", "content": request}]})
    return json.dumps({"agent": "kana", "answer": final_text(result), "trace": extract_tool_trace(result)}, ensure_ascii=False)

## 5. 확장 예제 실행

Supervisor에게는 `nana_agent`, `kana_agent` delegate tool만 제공한다.

In [4]:
# 실행
supervisor = create_agent(
    model=make_model(1000),
    tools=[nana_agent, kana_agent],
    middleware=[DisableParallelToolCalls()],
    system_prompt=(
        "너는 카나메이트 supervisor다. 업무 tool을 직접 처리하지 말고 delegate tool만 호출한다. "
        "모든 회의 시간 결정 요청에서 반드시 먼저 nana_agent를 호출해 내 개인 일정 충돌을 확인하고, "
        "그 다음 반드시 kana_agent를 호출해 팀원 A/B/C의 이전 대화 기반 후보만 확인한다. "
        "kana_agent에는 Nana가 확인한 개인 일정 내용을 전달하지 않는다. "
        "두 tool_result를 모두 받은 뒤 supervisor가 개인 일정과 팀원 후보를 종합해 최종 답변을 작성한다."
    ),
)

final_request = "팀원 A/B/C와 다음 주 회의 시간을 잡아줘. 내 개인 일정도 확인하고, 팀원들의 이전 대화 기록도 확인해서 최종 시간을 정해줘."
supervisor_result = supervisor.invoke({"messages": [{"role": "user", "content": final_request}]})
supervisor_answer = final_text(supervisor_result)
supervisor_trace = extract_tool_trace(supervisor_result)
delegate_payloads = extract_delegate_payloads(supervisor_trace)
supervisor_run = {
    "request": final_request,
    "answer": supervisor_answer,
    "trace": supervisor_trace,
    "delegate_payloads": delegate_payloads,
}

print(supervisor_answer)
show_json(supervisor_trace)
show_json({"delegate_payloads": delegate_payloads})


다음 주 회의 시간은 팀원 A, B, C 모두 가능한 화요일 15:00로 정하는 것이 좋겠습니다. 다만, 개인 일정 중 5월 20일(화) 10시부터 11시까지 '개인 운동' 일정이 있으니 이 시간과 겹치지 않는 점도 확인되었습니다. 따라서 다음 주 화요일 15:00에 회의를 잡도록 하겠습니다.
[
  {
    "event": "tool_call",
    "tool_name": "nana_agent",
    "arguments": {
      "request": "다음 주 내 개인 일정 확인 요청"
    }
  },
  {
    "event": "tool_result",
    "tool_name": "nana_agent",
    "content": "{\"agent\": \"nana\", \"answer\": \"다음 주 개인 일정 중에는 5월 20일 10시부터 11시까지 '개인 운동' 일정이 있습니다. 다른 일정 확인이나 회의 요청이 있으시면 말씀해 주세요.\", \"trace\": [{\"event\": \"tool_call\", \"tool_name\": \"personal_list_schedules\", \"arguments\": {\"date_range\": \"next_week\"}}, {\"event\": \"tool_result\", \"tool_name\": \"personal_list_schedules\", \"content\": \"{\\\"date_range\\\": \\\"next_week\\\", \\\"schedules\\\": [{\\\"title\\\": \\\"개인 운동\\\", \\\"date\\\": \\\"2026-05-20\\\", \\\"start_time\\\": \\\"10:00\\\", \\\"end_time\\\": \\\"11:00\\\"}]}\"}]}"
  },
  {
    "event": "tool_call",
    "tool_name": "kana_agent",
    "arguments"

### LLM-as-a-judge 검증 방식

기존 `assert` 검증은 delegate tool 호출 여부나 특정 문자열 포함 여부처럼 겉으로 드러난 형태를 빠르게 확인할 수 있지만, LLM 답변이 요구사항의 의미를 제대로 지켰는지 판단하기에는 취약하다. 여기서는 실행 결과 전체를 평가자 LLM에게 넘겨 의미 기준으로 검증한다.

평가자 LLM은 `supervisor_run`에 담긴 `supervisor_trace`, `delegate_payloads`, 최종 supervisor 답변을 근거로 다음 항목을 판정한다.

- supervisor가 업무 tool을 직접 처리하지 않고 `nana_agent`와 `kana_agent`에 위임했는가
- Nana 호출과 결과 수신 이후 Kana 호출과 결과 수신이 이어졌는가
- Nana가 `personal_list_schedules`로 사용자의 개인 일정을 확인했는가
- Kana가 팀원 A/B/C의 이전 대화 기반 tool들을 사용해 최종 후보를 결정했는가
- Kana가 사용자의 개인 일정 충돌 여부까지 판단하는 역할 침범을 하지 않았는가. 즉, Kana가 개인 일정 충돌을 판단하지 않은 것은 통과 근거다.
- 최종 supervisor 답변이 Nana의 개인 일정 확인 결과와 Kana의 팀원 가능 시간 판단을 함께 반영했는가

아래 검증 셀은 별도 `deterministic_checks`나 Pydantic judge schema를 두지 않는다. 평가 기준과 판정 규칙을 judge prompt에 모아 두고, 파이썬 코드는 judge 응답의 첫 줄이 `PASS`인지 확인하는 얇은 가드레일만 맡는다.


![DL_test](./imgs/DL_test.jpg)

![LLM_test](./imgs/LLM_test.jpg)

![LLM_test_prompt](./imgs/LLM_test_prompt.jpg)

In [5]:
# 검증: LLM-as-a-judge
import json

current_run = globals().get("supervisor_run")
judge_model_factory = globals().get("make_model")
judge_agent_factory = globals().get("create_agent")
final_text_reader = globals().get("final_text")
if current_run is None:
    raise RuntimeError("먼저 '5. 확장 예제 실행' 셀을 실행해 supervisor_run을 만든 뒤 검증 셀을 실행하세요.")
if judge_model_factory is None:
    raise RuntimeError("먼저 '1. 준비' 셀을 실행해 make_model을 정의하세요.")
if judge_agent_factory is None or final_text_reader is None:
    raise RuntimeError("먼저 '1. 준비' 셀을 실행해 create_agent와 final_text를 정의하세요.")

judge_context = {
    "request": current_run["request"],
    "supervisor_answer": current_run["answer"],
    "supervisor_trace": current_run["trace"],
    "delegate_payloads": current_run["delegate_payloads"],
}

judge_prompt = f"""
너는 multi-agent 일정 조율 시스템을 평가하는 엄격하지만 공정한 LLM judge다.
아래 실행 결과 JSON만 근거로 supervisor/sub-agent 동작이 요구사항을 만족했는지 평가한다.

반드시 다음 절차로 평가한다.
1. supervisor_trace에서 supervisor가 호출한 tool_call 순서가 nana_agent -> kana_agent인지 확인한다.
2. supervisor_trace에서 각 delegate tool_result가 같은 순서로 돌아왔는지 확인한다.
3. delegate_payloads에서 Nana의 trace를 찾아 personal_list_schedules tool_call이 있는지 확인한다.
4. delegate_payloads에서 Kana의 trace를 찾아 search_previous_conversations -> extract_schedules_from_history -> decide_final_slot 순서의 tool_call이 있는지 확인한다.
5. Kana가 personal_list_schedules를 호출하지 않았고, 답변에서 사용자 개인 일정 충돌 여부를 판단하거나 '개인 일정과 충돌하지 않는다'고 단정하지 않았는지 확인한다.
6. supervisor_answer가 Nana의 개인 일정 확인 결과와 Kana의 팀원 가능 시간 후보를 함께 반영해 최종 시간을 제안했는지 확인한다.

판정 규칙:
- 실행 결과 JSON 밖의 추측은 사용하지 않는다.
- 단순 문자열 포함 여부가 아니라 의미를 기준으로 판단한다.
- 한국어 표현이 조금 달라도 같은 의미면 허용한다.
- 특히 'Kana가 개인 일정 충돌 여부를 판단하지 않음'은 실패 사유가 아니라 평가 기준 5의 통과 사유다.
- 위 6개 기준을 모두 만족하면 첫 줄에 PASS를 쓴다.
- 하나라도 만족하지 않으면 첫 줄에 FAIL을 쓰고, 바로 아래에 실패 기준을 짧게 쓴다.
- PASS/FAIL 뒤에는 2~4문장으로 근거를 요약한다.

실행 결과 JSON:
{json.dumps(judge_context, ensure_ascii=False, indent=2, default=str)}
"""

judge_agent = judge_agent_factory(
    model=judge_model_factory(1000),
    tools=[],
    system_prompt="너는 에이전트 실행 trace를 평가하는 LLM-as-a-judge다. 주어진 JSON만 근거로 판정하고 첫 줄은 반드시 PASS 또는 FAIL로 시작한다.",
)
judge_result = judge_agent.invoke({"messages": [{"role": "user", "content": judge_prompt}]})
judge_response = final_text_reader(judge_result).strip()

print(judge_response)
assert judge_response.startswith("PASS"), judge_response


PASS  
supervisor_trace에서 nana_agent 호출 후 kana_agent 호출 순서와 tool_result 순서가 일치하며, delegate_payloads에서 Nana는 personal_list_schedules를 호출했고 Kana는 search_previous_conversations, extract_schedules_from_history, decide_final_slot 순서로 호출했다. Kana는 개인 일정 충돌 여부를 판단하지 않았고, supervisor_answer는 Nana의 개인 일정 확인 결과와 Kana의 팀원 가능 시간 후보를 모두 반영해 최종 시간을 제안했다. 모든 평가 기준을 충족한다.


## 6. 회고

확인 질문:

1. Supervisor가 직접 `personal_list_schedules`나 MCP 검색 tool을 호출하지 않는 이유는 무엇인가?
2. Nana와 Kana의 책임은 어떤 기준으로 나뉘는가?
3. 최종 일정 결정에서 가능한 시간 후보와 선택 이유를 함께 보여줘야 하는 이유는 무엇인가?

작은 응용 과제: 팀원 한 명이 선택된 시간에 불가능하다고 가정하고 `decide_final_slot` 결과와 supervisor 답변이 어떻게 달라져야 하는지 작성한다.